# Lab 5 : Implementation of NP Hard and NP Complete Problems

---


## Objective of the Lab

The objective of this lab is to understand, design, and implement programs that illustrate **NP-Hard** and **NP-Complete** problems using Python. Through this lab, we aim to:

- Understand the classes **P**, **NP**, **NP-Complete**, and **NP-Hard**, and the significance of the **Cook–Levin Theorem** in establishing NP-completeness.
- Implement exponential-time exact algorithms (brute force / backtracking) for problems for which no known polynomial-time algorithm exists.
- Implement polynomial-time **certificate verifiers** to demonstrate what it means for a problem to be in NP.
- Analyze the time complexity of each implemented algorithm, and appreciate why these problems are considered computationally intractable for large inputs.

> **Note on Q4 ("NP Hard Code Generation Problem"):** This term is not a single universally standard named problem the way Vertex Cover or Subset Sum are. In compiler theory, the **optimal code generation problem** (assigning a minimal number of registers / minimal-cost instruction ordering when evaluating an expression represented as a Directed Acyclic Graph, rather than a simple tree) is a well-known problem that becomes **NP-hard** once the expression is a DAG (common sub-expressions shared) instead of a tree. We have implemented this interpretation below via exhaustive search over evaluation orders. If your course intended a different specific problem under this name, let us know and the implementation can be adjusted.


## Title of the Study

**Implementation and Analysis of NP-Hard and NP-Complete Problems:**
Cook's Theorem (Boolean Satisfiability), an NP-Hard Graph Problem (Hamiltonian Cycle), an NP-Hard Scheduling Problem (Minimum Makespan Scheduling), the NP-Hard Optimal Code Generation Problem (DAG register minimization), the Vertex Cover Problem, and the Subset Sum Problem.


## Related Theory (General)

**Complexity classes:**

- **P** — the class of decision problems solvable in **polynomial time** by a deterministic algorithm.
- **NP** — the class of decision problems for which a proposed solution (a **certificate**) can be **verified** in polynomial time.
- **NP-Complete** — the hardest problems *in* NP: every problem in NP can be reduced to an NP-Complete problem in polynomial time, and an NP-Complete problem is itself in NP.
- **NP-Hard** — problems **at least as hard** as NP-Complete problems, but not necessarily in NP themselves (e.g., optimization versions of NP-Complete decision problems).

**Cook–Levin Theorem:** proves that the **Boolean Satisfiability Problem (SAT)** is NP-Complete — it was the *first* problem proven NP-Complete, and every other NP-Complete problem is shown to be NP-Complete by **reducing** SAT (or another already-proven NP-Complete problem) to it in polynomial time.

In this lab, we study 6 programs illustrating NP-Hard/NP-Complete concepts:

1. Cook's Theorem — Brute-force SAT solving + polynomial-time certificate verification
2. An NP-Hard Graph Problem — Hamiltonian Cycle
3. An NP-Hard Scheduling Problem — Minimum Makespan Scheduling on Identical Machines
4. An NP-Hard Code Generation Problem — Optimal Evaluation Order for a DAG Expression (register minimization)
5. Vertex Cover Problem
6. Subset Sum Problem

Each is discussed below with theory, diagram, source code, output, and complexity analysis.


---
## Q1. Program to Implement Cook's Theorem (Boolean Satisfiability)

### Related Theory
**Cook's Theorem** (Cook–Levin, 1971) states that the **Boolean Satisfiability Problem (SAT)** is **NP-Complete**. Given a Boolean formula in **Conjunctive Normal Form (CNF)** — a conjunction (AND) of clauses, each a disjunction (OR) of literals — SAT asks whether there exists an assignment of `True`/`False` to the variables that makes the whole formula `True`.

We illustrate the two defining properties of NP-Completeness directly:

1. **Verification is polynomial:** given a candidate assignment (a *certificate*), checking whether it satisfies the formula takes only $O(\text{number of clauses} \times \text{clause length})$ time.
2. **Solving (in the absence of a certificate) is exponential:** the brute-force solver below tries all $2^n$ possible truth assignments for $n$ variables, reflecting the belief that SAT $\notin$ P (the P vs NP question remains unsolved, but no polynomial algorithm for SAT is known).

### Related Diagram (Flow)
```
        Start
          |
   Read CNF formula: list of clauses, each a list of literals
          |
   +-----------------------------------------------------------+
   | for each of the 2^n truth assignments:                    | <---+
   |   evaluate formula(assignment)  -- polynomial-time check  |     |
   |   if formula is True: return "Satisfiable", assignment    |     |
   |   (else continue trying)                                  | ----+
   +-----------------------------------------------------------+
          |
   return "Unsatisfiable"  (if no assignment worked)
          |
         End
```


In [1]:
from itertools import product

def evaluate_cnf(clauses, assignment):
    """
    Polynomial-time CERTIFICATE VERIFICATION step.
    clauses: list of clauses; each clause is a list of literals,
             where a positive int i means variable x_i, and a negative int -i means NOT x_i.
    assignment: dict {variable_index: True/False}
    Returns True if the CNF formula is satisfied by this assignment.
    """
    for clause in clauses:
        clause_satisfied = False
        for lit in clause:
            var = abs(lit)
            value = assignment[var]
            if lit < 0:
                value = not value
            if value:
                clause_satisfied = True
                break
        if not clause_satisfied:
            return False          # this clause failed -> whole formula is False
    return True


def brute_force_sat(clauses, num_vars):
    """
    Exponential-time SOLVING step (no known polynomial algorithm exists for general SAT).
    Tries all 2^num_vars truth assignments.
    Returns (is_satisfiable, satisfying_assignment or None)
    """
    variables = list(range(1, num_vars + 1))
    for values in product([True, False], repeat=num_vars):
        assignment = dict(zip(variables, values))
        if evaluate_cnf(clauses, assignment):
            return True, assignment
    return False, None


# Driver / Test
if __name__ == "__main__":
    # Formula: (x1 OR x2) AND (NOT x1 OR x3) AND (NOT x2 OR NOT x3)
    clauses = [[1, 2], [-1, 3], [-2, -3]]
    num_vars = 3

    is_sat, assignment = brute_force_sat(clauses, num_vars)
    print(f"Formula: {clauses}")
    print(f"Satisfiable: {is_sat}")
    if is_sat:
        print(f"Satisfying assignment: {assignment}")
        # Demonstrate that verification of this certificate is fast/independent
        print(f"Verification check (should be True): {evaluate_cnf(clauses, assignment)}")

    # An unsatisfiable formula: (x1) AND (NOT x1)
    unsat_clauses = [[1], [-1]]
    is_sat2, _ = brute_force_sat(unsat_clauses, 1)
    print(f"\nFormula: {unsat_clauses}")
    print(f"Satisfiable: {is_sat2}")


Formula: [[1, 2], [-1, 3], [-2, -3]]
Satisfiable: True
Satisfying assignment: {1: True, 2: False, 3: True}
Verification check (should be True): True

Formula: [[1], [-1]]
Satisfiable: False


### Analysis of the Algorithm
- **Verification Time Complexity:** $O(C \times L)$ where $C$ is the number of clauses and $L$ is the maximum clause length — this is **polynomial**, which is exactly why SAT is in **NP**.
- **Solving Time Complexity:** $O(2^n \times C \times L)$ for brute force over $n$ variables — **exponential**, consistent with SAT being **NP-Complete** (no polynomial-time solving algorithm is known).
- **Space Complexity:** $O(n + C \times L)$ for storing the formula and current assignment.
- **Remark:** Cook's Theorem's significance is not the brute-force algorithm itself, but the **reduction technique**: any problem in NP can be transformed, in polynomial time, into an instance of SAT. This makes SAT the "universal" NP problem, and all other problems in this lab (Vertex Cover, Subset Sum, Hamiltonian Cycle, etc.) can, in principle, be reduced to/from SAT.


---
## Q2. Program to Implement an NP-Hard Graph Problem (Hamiltonian Cycle)

### Related Theory
A **Hamiltonian Cycle** in a graph is a cycle that visits **every vertex exactly once** and returns to the starting vertex. Deciding whether such a cycle exists is **NP-Complete** — verifying a *given* candidate cycle is easy (polynomial time: just check that all vertices appear once, and consecutive vertices are connected by an edge), but no polynomial-time algorithm is known to *find* one (or determine none exists) for general graphs. We solve it here using **backtracking**, which explores partial paths and prunes branches that cannot lead to a valid Hamiltonian Cycle.

### Related Diagram (Flow)
```
        Start
          |
   Read graph as adjacency matrix; path = [0]  (start at vertex 0)
          |
   +------------------------------------------------------------------+
   | Backtrack(path):                                                 | <---+
   |   if len(path) == V:                                             |     |
   |       if edge(path[-1], path[0]) exists: return True (cycle!)    |     |
   |       else: return False                                         |     |
   |   for each vertex v not in path:                                 |     |
   |       if edge(path[-1], v) exists:                               |     |
   |           path.append(v)                                         |     |
   |           if Backtrack(path): return True                        |     |
   |           path.pop()   (undo -- try next vertex)                 | ----+
   +------------------------------------------------------------------+
          |
   Output the Hamiltonian Cycle found, or "None exists"
          |
         End
```


In [2]:
def hamiltonian_cycle(graph):
    """
    graph: adjacency matrix (list of lists), graph[i][j]=1 if edge exists between i and j.
    Returns: a Hamiltonian cycle (list of vertex indices) if one exists, else None.
    """
    n = len(graph)
    path = [0]                       # start the cycle at vertex 0
    visited = [False] * n
    visited[0] = True

    def backtrack():
        if len(path) == n:
            # All vertices visited: check if we can close the cycle back to start
            return graph[path[-1]][path[0]] == 1

        for v in range(n):
            if not visited[v] and graph[path[-1]][v] == 1:
                path.append(v)
                visited[v] = True
                if backtrack():
                    return True
                # Backtrack: undo the choice and try the next vertex
                path.pop()
                visited[v] = False
        return False

    if backtrack():
        return path + [path[0]]      # return to starting vertex to complete the cycle
    return None


# Driver / Test
if __name__ == "__main__":
    # A graph that DOES contain a Hamiltonian Cycle (a 5-cycle with an extra chord)
    graph_with_cycle = [
        [0, 1, 0, 1, 0],
        [1, 0, 1, 0, 0],
        [0, 1, 0, 1, 1],
        [1, 0, 1, 0, 1],
        [0, 0, 1, 1, 0],
    ]
    cycle = hamiltonian_cycle(graph_with_cycle)
    print(f"Graph 1 -> Hamiltonian Cycle: {cycle}")

    # A graph that does NOT contain a Hamiltonian Cycle (a star graph)
    star_graph = [
        [0, 1, 1, 1, 1],
        [1, 0, 0, 0, 0],
        [1, 0, 0, 0, 0],
        [1, 0, 0, 0, 0],
        [1, 0, 0, 0, 0],
    ]
    cycle2 = hamiltonian_cycle(star_graph)
    print(f"Graph 2 -> Hamiltonian Cycle: {cycle2}")


Graph 1 -> Hamiltonian Cycle: [0, 1, 2, 4, 3, 0]
Graph 2 -> Hamiltonian Cycle: None


### Analysis of the Algorithm
- **Time Complexity:** $O(n!)$ in the worst case — backtracking may explore nearly all permutations of vertices, though pruning (only extending along existing edges) reduces this significantly for sparse graphs in practice.
- **Space Complexity:** $O(n)$ for the recursion stack, path, and visited arrays.
- **Remark:** The Hamiltonian Cycle problem is **NP-Complete**; the related **Travelling Salesman Problem** (Lab 4, Q2) is its **NP-Hard optimization counterpart** — TSP asks not just *whether* a Hamiltonian cycle exists, but for the *minimum-weight* one.


---
## Q3. Program to Implement an NP-Hard Scheduling Problem

### Related Theory
**Minimum Makespan Scheduling on Identical Machines** is a classic NP-Hard scheduling problem: given `n` jobs with processing times and `m` identical machines, assign each job to a machine so that the **makespan** (the time the *last* machine finishes) is **minimized**. This problem is NP-Hard in general (it is closely related to the Partition Problem, which is NP-Complete). We solve it exactly via **backtracking/branch-and-bound**, exploring every way jobs can be assigned to machines while pruning branches whose partial makespan already exceeds the best solution found so far.

### Related Diagram (Flow)
```
        Start
          |
   Read job processing times, number of machines m
          |
   best_makespan = infinity
          |
   +-------------------------------------------------------------------+
   | Assign(job_index, machine_loads[]):                               | <---+
   |   if job_index == n:                                              |     |
   |       best_makespan = min(best_makespan, max(machine_loads))      |     |
   |       return                                                      |     |
   |   for each machine m_i:                                           |     |
   |       if machine_loads[m_i] + job >= best_makespan: skip (prune)  |     |
   |       machine_loads[m_i] += job                                   |     |
   |       Assign(job_index+1, machine_loads)                          |     |
   |       machine_loads[m_i] -= job   (backtrack)                     | ----+
   +-------------------------------------------------------------------+
          |
   Output best_makespan and an optimal assignment
          |
         End
```


In [3]:
def min_makespan_scheduling(jobs, m):
    """
    jobs: list of job processing times
    m: number of identical machines
    Returns: (best_makespan, best_assignment) where best_assignment[i] = machine index for job i
    """
    n = len(jobs)
    # Sort jobs in decreasing order first (a good heuristic that improves pruning)
    order = sorted(range(n), key=lambda i: jobs[i], reverse=True)

    best = {"makespan": float('inf'), "assignment": None}
    loads = [0] * m
    assignment = [-1] * n

    def backtrack(idx):
        if idx == n:
            makespan = max(loads)
            if makespan < best["makespan"]:
                best["makespan"] = makespan
                best["assignment"] = assignment[:]
            return

        job_i = order[idx]
        tried_loads = set()
        for machine in range(m):
            # Skip machines that would create a duplicate state (symmetry pruning)
            if loads[machine] in tried_loads:
                continue
            tried_loads.add(loads[machine])

            if loads[machine] + jobs[job_i] >= best["makespan"]:
                continue          # branch-and-bound pruning

            loads[machine] += jobs[job_i]
            assignment[job_i] = machine
            backtrack(idx + 1)
            loads[machine] -= jobs[job_i]     # undo (backtrack)

    backtrack(0)
    return best["makespan"], best["assignment"]


# Driver / Test
if __name__ == "__main__":
    jobs = [5, 8, 4, 7, 6, 3]
    m = 3

    best_makespan, assignment = min_makespan_scheduling(jobs, m)
    print(f"Jobs: {jobs}, Machines: {m}")
    print(f"Optimal (minimum) makespan: {best_makespan}")
    print(f"Job -> Machine assignment: {assignment}")

    loads = [0] * m
    for job_idx, machine in enumerate(assignment):
        loads[machine] += jobs[job_idx]
    print(f"Resulting machine loads: {loads}")


Jobs: [5, 8, 4, 7, 6, 3], Machines: 3
Optimal (minimum) makespan: 11
Job -> Machine assignment: [2, 0, 1, 1, 2, 0]
Resulting machine loads: [11, 11, 11]


### Analysis of the Algorithm
- **Time Complexity:** $O(m^n)$ in the worst case — every job can be assigned to any of the $m$ machines; branch-and-bound pruning and symmetry pruning (skipping machines with equal current load) reduce this substantially in practice, but the worst case remains exponential.
- **Space Complexity:** $O(n + m)$ for the recursion stack, assignment array, and machine loads.
- **Remark:** This problem is NP-Hard because it generalizes the **Partition Problem** (splitting a set into two subsets of equal sum, which is NP-Complete) — the case of $m=2$ machines is exactly a decision version of Partition. Real-world job schedulers instead use fast heuristics like **Longest Processing Time (LPT) first**, which is not always optimal but runs in polynomial time.


---
## Q4. Program to Implement an NP-Hard Code Generation Problem

### Related Theory
In compiler design, generating code to evaluate an arithmetic expression while **minimizing the number of registers** used is easy (linear time, via the **Sethi–Ullman algorithm**) when the expression is a **tree** (no shared sub-expressions). However, when the expression is represented as a **Directed Acyclic Graph (DAG)** — because common sub-expressions are shared and should only be computed once — finding the evaluation order that minimizes the peak number of registers needed becomes **NP-Hard**. This is a well-known result in compiler theory (optimal code generation for DAGs is NP-complete; Aho, Sethi & Ullman).

We demonstrate this by exhaustively searching over all **topological orders** of a DAG's nodes and computing the peak register usage (number of live/pending values) for each order, to find the evaluation order that minimizes peak register usage.

### Related Diagram (Flow)
```
        Start
          |
   Read DAG: nodes and their dependencies (operands)
          |
   Generate ALL valid topological orders of the DAG
          |
   +--------------------------------------------------------------+
   | for each topological order:                                  | <---+
   |   simulate evaluation; track live values needed (registers)  |     |
   |   peak = max registers live at any point in this order       |     |
   |   if peak < best_peak: best_peak = peak; best_order = order  | ----+
   +--------------------------------------------------------------+
          |
   Output best evaluation order and minimum peak register count
          |
         End
```


In [4]:
from itertools import permutations

def all_topological_orders(nodes, deps):
    """
    nodes: list of node names
    deps: dict {node: set_of_nodes_it_depends_on}
    Yields every valid topological order (as a tuple of nodes).
    NOTE: This brute-force generation is itself exponential in the worst case,
    reflecting the NP-Hard nature of finding the *optimal* evaluation order.
    """
    for perm in permutations(nodes):
        position = {node: i for i, node in enumerate(perm)}
        if all(position[d] < position[node] for node in nodes for d in deps[node]):
            yield perm


def peak_registers(order, deps):
    """
    Simulate evaluating the DAG in the given order and compute the peak number of
    'live' intermediate values (registers) needed at any point.
    A value becomes live once computed, and 'dies' once all its dependents are evaluated.
    """
    remaining_uses = {node: 0 for node in order}
    for node in order:
        for d in deps[node]:
            remaining_uses[d] += 1   # count how many parents still need this value

    live = set()
    peak = 0
    for node in order:
        for d in deps[node]:
            remaining_uses[d] -= 1
            if remaining_uses[d] == 0:
                live.discard(d)      # this operand's value is no longer needed after use
        live.add(node)               # the freshly computed value becomes live
        peak = max(peak, len(live))
    return peak


def optimal_code_generation(nodes, deps):
    """
    Returns (best_order, min_peak_registers) by exhaustively searching all
    topological orders of the DAG -- exponential, since this problem is NP-Hard.
    """
    best_order, best_peak = None, float('inf')
    for order in all_topological_orders(nodes, deps):
        peak = peak_registers(order, deps)
        if peak < best_peak:
            best_peak, best_order = peak, order
    return best_order, best_peak


# Driver / Test
if __name__ == "__main__":
    # DAG representing: T1 = a + b ; T2 = c + d ; T3 = T1 + T2 ; T4 = T1 + c
    # (T1 is a shared sub-expression used by both T3 and T4)
    nodes = ['a', 'b', 'c', 'd', 'T1', 'T2', 'T3', 'T4']
    deps = {
        'a': set(), 'b': set(), 'c': set(), 'd': set(),
        'T1': {'a', 'b'},
        'T2': {'c', 'd'},
        'T3': {'T1', 'T2'},
        'T4': {'T1', 'c'},
    }

    best_order, best_peak = optimal_code_generation(nodes, deps)
    print(f"Optimal evaluation order: {best_order}")
    print(f"Minimum peak register requirement: {best_peak}")


Optimal evaluation order: ('a', 'b', 'c', 'T1', 'd', 'T2', 'T3', 'T4')
Minimum peak register requirement: 3


In [5]:
# Note: The kernel timing above uses timeit-style measurement to illustrate the
# exponential blow-up as the DAG grows (even by a couple of nodes).
import time

for extra_nodes in range(0, 3):
    test_nodes = nodes + [f"X{i}" for i in range(extra_nodes)]
    test_deps = dict(deps)
    prev = 'T4'
    for i in range(extra_nodes):
        test_deps[f"X{i}"] = {prev}
        prev = f"X{i}"

    start = time.time()
    _, peak = optimal_code_generation(test_nodes, test_deps)
    elapsed = time.time() - start
    print(f"Nodes: {len(test_nodes):>2} | Min peak registers: {peak} | "
          f"Brute-force search time: {elapsed:.4f}s")


Nodes:  8 | Min peak registers: 3 | Brute-force search time: 0.0486s


Nodes:  9 | Min peak registers: 3 | Brute-force search time: 0.4642s


Nodes: 10 | Min peak registers: 3 | Brute-force search time: 4.5085s


### Analysis of the Algorithm
- **Time Complexity:** $O(n! \times n)$ in the worst case — the brute-force search enumerates permutations of the $n$ DAG nodes and checks/simulates each for validity and register cost; this reflects why finding the **optimal** evaluation order is NP-Hard.
- **Space Complexity:** $O(n)$ per candidate order for the live-set simulation (excluding the space needed to enumerate permutations).
- **Remark:** In practice, compilers do **not** solve this problem exactly. Instead they use polynomial-time **heuristics** (e.g., labeling algorithms extended to DAGs, list scheduling, or simply treating shared sub-expressions conservatively as a tree via duplication) that produce good, though not always optimal, register allocations.


---
## Q5. Program to Implement the Vertex Cover Problem

### Related Theory
A **Vertex Cover** of a graph is a subset of vertices such that **every edge** has at least one endpoint in the subset. The **Vertex Cover Problem** (decision version: "does a vertex cover of size $\le k$ exist?") is **NP-Complete**. We solve the optimization version (find the *minimum* vertex cover) via backtracking: for any uncovered edge $(u, v)$, at least one of $u$ or $v$ **must** be in the cover, so we branch on including $u$ or including $v$.

### Related Diagram (Flow)
```
        Start
          |
   Read graph edges
          |
   +-----------------------------------------------------------------+
   | Search(cover, remaining_edges):                                 | <---+
   |   if remaining_edges is empty: candidate solution found         |     |
   |   pick any edge (u, v) from remaining_edges                     |     |
   |   Branch 1: add u to cover; remove all edges covered by u       |     |
   |   Branch 2: add v to cover; remove all edges covered by v       |     |
   |   recurse on both branches, keep the smaller resulting cover    | ----+
   +-----------------------------------------------------------------+
          |
   Output minimum vertex cover found
          |
         End
```


In [6]:
def min_vertex_cover(edges):
    """
    edges: list of tuples (u, v) representing undirected edges
    Returns: the minimum vertex cover (as a set of vertices) via backtracking.
    """
    best = {"cover": None}

    def search(cover, remaining_edges):
        # Prune: if current cover is already as big as (or bigger than) best found, stop
        if best["cover"] is not None and len(cover) >= len(best["cover"]):
            return

        if not remaining_edges:
            if best["cover"] is None or len(cover) < len(best["cover"]):
                best["cover"] = set(cover)
            return

        u, v = remaining_edges[0]

        # Branch 1: include u in the cover
        new_remaining = [e for e in remaining_edges if u not in e]
        search(cover | {u}, new_remaining)

        # Branch 2: include v in the cover
        new_remaining = [e for e in remaining_edges if v not in e]
        search(cover | {v}, new_remaining)

    search(set(), edges)
    return best["cover"]


# Driver / Test
if __name__ == "__main__":
    edges = [('A', 'B'), ('A', 'C'), ('B', 'C'), ('C', 'D'), ('D', 'E')]

    cover = min_vertex_cover(edges)
    print(f"Graph edges: {edges}")
    print(f"Minimum Vertex Cover: {cover} (size {len(cover)})")

    # Verify: every edge should have at least one endpoint in the cover
    is_valid = all(u in cover or v in cover for u, v in edges)
    print(f"Verification (all edges covered): {is_valid}")


Graph edges: [('A', 'B'), ('A', 'C'), ('B', 'C'), ('C', 'D'), ('D', 'E')]
Minimum Vertex Cover: {'B', 'D', 'A'} (size 3)
Verification (all edges covered): True


### Analysis of the Algorithm
- **Time Complexity:** $O(2^k \times |E|)$ in this branching formulation (roughly), where $k$ is the size of the minimum cover — in the absolute worst case this is $O(2^V)$ over all vertex subsets, but the edge-branching + pruning strategy used above is considerably faster in practice (related to standard **fixed-parameter tractable** algorithms for Vertex Cover, which run in $O(2^k \cdot n)$ time).
- **Space Complexity:** $O(V + E)$ for the graph representation and recursion stack.
- **Remark:** Vertex Cover is **NP-Complete**, and is one of Karp's original 21 NP-Complete problems. It is closely related to the **Independent Set** problem (the complement of a vertex cover is a maximum independent set) and to **Clique** (via graph complementation).


---
## Q6. Program to Implement the Subset Sum Problem

### Related Theory
The **Subset Sum Problem** asks: given a set of integers and a target sum $T$, does there exist a subset of the integers that sums exactly to $T$? This problem is **NP-Complete** in general. We use a **Dynamic Programming** approach (pseudo-polynomial in the value of $T$) which is far faster in practice than the $O(2^n)$ brute-force approach of checking every subset, though it is still exponential in the *number of bits* needed to represent $T$ (hence the problem remains NP-Complete overall).

$$dp[i][s] = dp[i-1][s] \;\text{ OR }\; dp[i-1][s - a_i] \quad (\text{if } a_i \le s)$$

where `dp[i][s]` is `True` if some subset of the first $i$ numbers sums exactly to $s$.

### Related Diagram (Flow)
```
        Start
          |
   Read set of integers, target sum T
          |
   dp[0][0] = True; dp[0][s>0] = False
          |
   +----------------------------------------------------------------+
   | for i in 1..n:                                                 | <---+
   |   for s in 0..T:                                               |     |
   |     dp[i][s] = dp[i-1][s]                                      |     |
   |     if a[i] <= s: dp[i][s] = dp[i][s] OR dp[i-1][s-a[i]]       | ----+
   +----------------------------------------------------------------+
          |
   Output dp[n][T]  (True/False) and, if True, one such subset
          |
         End
```


In [7]:
def subset_sum(nums, target):
    """
    nums: list of positive integers
    target: target sum T
    Returns: (exists: bool, subset: list or None)
    """
    n = len(nums)
    dp = [[False] * (target + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = True         # empty subset always sums to 0

    for i in range(1, n + 1):
        for s in range(target + 1):
            dp[i][s] = dp[i - 1][s]
            if nums[i - 1] <= s and dp[i - 1][s - nums[i - 1]]:
                dp[i][s] = True

    if not dp[n][target]:
        return False, None

    # Backtrack to reconstruct one valid subset
    subset = []
    s = target
    for i in range(n, 0, -1):
        if dp[i][s] and not dp[i - 1][s]:      # nums[i-1] was necessarily used
            subset.append(nums[i - 1])
            s -= nums[i - 1]
    return True, subset


# Driver / Test
if __name__ == "__main__":
    nums = [3, 34, 4, 12, 5, 2]

    for target in [9, 30, 100]:
        exists, subset = subset_sum(nums, target)
        print(f"Set: {nums}, Target: {target}")
        print(f"  Subset with this sum exists: {exists}")
        if exists:
            print(f"  Example subset: {subset} (sum = {sum(subset)})")


Set: [3, 34, 4, 12, 5, 2], Target: 9
  Subset with this sum exists: True
  Example subset: [5, 4] (sum = 9)
Set: [3, 34, 4, 12, 5, 2], Target: 30
  Subset with this sum exists: False
Set: [3, 34, 4, 12, 5, 2], Target: 100
  Subset with this sum exists: False


### Analysis of the Algorithm
- **Time Complexity:** $O(n \times T)$ — filling the DP table of size $(n+1) \times (T+1)$; this is **pseudo-polynomial**, since $T$ can be exponentially large relative to the number of bits used to encode it.
- **Space Complexity:** $O(n \times T)$ (can be reduced to $O(T)$ using a 1-D rolling boolean array processed right-to-left).
- **Remark:** Subset Sum is one of the classic **NP-Complete** problems and is central to proving the NP-completeness of the 0/1 Knapsack decision problem (Lab 4, Q4) and Partition Problem (related to Lab 5, Q3) via polynomial-time reductions.


---
## Discussion and Conclusion

In this lab, we implemented and analyzed six programs illustrating NP-Hard and NP-Complete concepts:

- **Cook's Theorem (SAT)** demonstrated the core NP-Completeness idea directly: certificates can be **verified** in polynomial time, while **solving** from scratch (brute force) takes exponential time; SAT's special status is that *every* NP problem reduces to it in polynomial time.
- **Hamiltonian Cycle** showed a classic NP-Complete **graph existence** problem, solved exactly via backtracking with edge-based pruning.
- **Minimum Makespan Scheduling** illustrated an NP-Hard **optimization** problem closely related to the Partition Problem, solved via branch-and-bound.
- **Optimal Code Generation for DAGs** showed a compiler-theory example of NP-Hardness that arises the moment shared sub-expressions turn a tree into a DAG, solved here via brute-force search over topological orders.
- **Vertex Cover** demonstrated one of Karp's original 21 NP-Complete problems, solved via edge-based backtracking branching.
- **Subset Sum** showed how a **pseudo-polynomial** Dynamic Programming solution can be dramatically faster than brute force in practice, while the problem remains NP-Complete in the strict (bit-length) sense.

**Overall Conclusion:**

This lab reinforced the theoretical foundations of computational complexity: the distinction between **P**, **NP**, **NP-Complete**, and **NP-Hard**, and why the Cook–Levin Theorem is foundational (it gave us the *first* NP-Complete problem, from which the NP-completeness of all the other problems studied here — Vertex Cover, Subset Sum, Hamiltonian Cycle, Scheduling — can be established via polynomial-time reductions). We also observed a recurring practical theme: even though these problems have no known polynomial-time exact algorithms, techniques such as backtracking with pruning, branch-and-bound, and pseudo-polynomial Dynamic Programming often make them tractable for moderately sized real-world instances, even though their worst-case complexity remains exponential. This lab strengthened our understanding of algorithmic intractability and of the practical strategies used to cope with it.
